# Train Qwen3 1.7B LoRA Adapters

Train one adapter on theorem-explicit reasoning and one adapter on theorem-implicit reasoning.

In [ ]:
!pip install -q -U "mlx-lm[train]" pandas tqdm

In [ ]:
from pathlib import Path
import shlex
import subprocess
import sys

In [ ]:
PROJECT_ROOT = Path("/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune")
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

In [ ]:
MODEL_NAME = "Qwen/Qwen3-1.7B-MLX-bf16"
DATA_ROOT = PROJECT_ROOT / "training_eval" / "fine_tune_qwen1_7B" / "lora" / "data"
RESULT_ROOT = PROJECT_ROOT / "results" / "fine_tunes" / "qwen3_1_7b_lora"

In [ ]:
ITERS = 80
BATCH_SIZE = 1
GRAD_ACCUMULATION_STEPS = 8
NUM_LAYERS = 4
LEARNING_RATE = "1e-5"
SKIP_IF_ADAPTER_EXISTS = True

In [ ]:
EXPERIMENTS = {
    "explicit_theorems": {
        "data_dir": DATA_ROOT / "explicit_theorems",
        "adapter_path": RESULT_ROOT / "explicit_theorems" / "adapters",
    },
    "implicit_theorems": {
        "data_dir": DATA_ROOT / "implicit_theorems",
        "adapter_path": RESULT_ROOT / "implicit_theorems" / "adapters",
    },
}

In [ ]:
def make_lora_command(data_dir, adapter_path):
    return [
        sys.executable,
        "-m",
        "mlx_lm.lora",
        "--model",
        MODEL_NAME,
        "--train",
        "--data",
        str(data_dir),
        "--adapter-path",
        str(adapter_path),
        "--iters",
        str(ITERS),
        "--batch-size",
        str(BATCH_SIZE),
        "--grad-accumulation-steps",
        str(GRAD_ACCUMULATION_STEPS),
        "--num-layers",
        str(NUM_LAYERS),
        "--learning-rate",
        LEARNING_RATE,
        "--mask-prompt",
        "--grad-checkpoint",
    ]

In [ ]:
def adapter_exists(adapter_path):
    return (adapter_path / "adapters.safetensors").exists() or (adapter_path / "adapters.npz").exists()

In [ ]:
def run_lora_training(name, config):
    if SKIP_IF_ADAPTER_EXISTS and adapter_exists(config["adapter_path"]):
        print(f"Skipping {name}; adapter already exists at {config['adapter_path']}")
        return

    config["adapter_path"].mkdir(parents=True, exist_ok=True)
    command = make_lora_command(config["data_dir"], config["adapter_path"])
    print(shlex.join(command))
    subprocess.run(command, check=True)

## Smoke Train

Run this first. It trains only the explicit adapter with the current short `ITERS` setting. If this works, run both adapters below.

In [ ]:
RUN_SMOKE = False

if RUN_SMOKE:
    run_lora_training("explicit_theorems", EXPERIMENTS["explicit_theorems"])

## Full Pair

After the smoke train is fine, increase `ITERS` if desired, restart the notebook, and run this cell.

In [ ]:
RUN_TRAINING = True

if RUN_TRAINING:
    for name, config in EXPERIMENTS.items():
        run_lora_training(name, config)